In [29]:
# Import necessary libraries for geospatial data processing and progress tracking
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import box, Point
from scipy.spatial import cKDTree
from scipy.stats import gaussian_kde
from tqdm import tqdm
import sys

# Debug: Confirm that imports are successful
print("Debug: Libraries imported successfully.")

# Print the versions of Python and each imported module
print(f"Python version: {sys.version}")
print(f"os version: Part of Python standard library, version {sys.version}")
print(f"numpy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"geopandas version: {gpd.__version__}")
print(f"osmnx version: {ox.__version__}")

# Set the base directory for datasets in Kaggle
base_dir = r"/kaggle/input/eyds-base-dataset"  # Base directory for input datasets
sub_dir = r"/kaggle/working/"  # Submission directory for output files

# Debug: Print the directory paths to confirm they are set correctly
print(f"Debug: Base directory: {base_dir}")
print(f"Debug: Submission directory: {sub_dir}")

# Debug: List the files in the base directory to verify the presence of input files
print(f"Debug: Listing files in {base_dir}:")
try:
    files_in_dir = os.listdir(base_dir)
    print(files_in_dir)
except Exception as e:
    print(f"Error: Could not list files in {base_dir}. Error: {str(e)}")
    raise Exception(f"Failed to access directory {base_dir}")

# File paths for training and validation data
TRAIN_CSV = os.path.join(base_dir, "Training_data.csv")
VALIDATION_CSV = os.path.join(base_dir, "Validation_data.csv")

# Load training data (limited to 500 rows for faster processing)
print("Loading training data...")
print(f"Debug: The file {TRAIN_CSV} exists: {os.path.exists(TRAIN_CSV)}")
try:
    train_df = pd.read_csv(TRAIN_CSV)
except FileNotFoundError:
    print(f"Error: Training data file not found at {TRAIN_CSV}")
    raise Exception("Failed to load training dataset")

# Create GeoDataFrame for training data
train_gdf = gpd.GeoDataFrame(
    train_df,
    geometry=gpd.points_from_xy(train_df.Longitude, train_df.Latitude),
    crs="EPSG:4326"
)
print(f"Debug: Training GeoDataFrame shape: {train_gdf.shape}")
print(f"Debug: Training GeoDataFrame columns: {train_gdf.columns.tolist()}")

# Load validation data (limited to 500 rows for faster processing)
print("Loading validation data...")
print(f"Debug: The file {VALIDATION_CSV} exists: {os.path.exists(VALIDATION_CSV)}")
try:
    val_df = pd.read_csv(VALIDATION_CSV)
except FileNotFoundError:
    print(f"Error: Validation data file not found at {VALIDATION_CSV}")
    raise Exception("Failed to load validation dataset")

# Create GeoDataFrame for validation data
val_gdf = gpd.GeoDataFrame(
    val_df,
    geometry=gpd.points_from_xy(val_df.Longitude, val_df.Latitude),
    crs="EPSG:4326"
)
print(f"Debug: Validation GeoDataFrame shape: {val_gdf.shape}")
print(f"Debug: Validation GeoDataFrame columns: {val_gdf.columns.tolist()}")

# Create bounding box from training data
min_lon, min_lat = train_gdf.geometry.x.min(), train_gdf.geometry.y.min()
max_lon, max_lat = train_gdf.geometry.x.max(), train_gdf.geometry.y.max()
bbox_polygon = box(min_lon, min_lat, max_lon, max_lat)
# Convert to GeoSeries and set CRS
bbox_polygon = gpd.GeoSeries([bbox_polygon], crs="EPSG:4326").iloc[0]
print("Debug: Bounding box created from training data.")
print(f"Debug: Bounding box bounds: min_lon={min_lon}, min_lat={min_lat}, max_lon={max_lon}, max_lat={max_lat}")

# Define target CRS (metric)
target_crs = "EPSG:32618"

Debug: Libraries imported successfully.
Python version: 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
os version: Part of Python standard library, version 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
numpy version: 1.26.4
pandas version: 2.2.3
geopandas version: 0.14.4
osmnx version: 2.0.1
Debug: Base directory: /kaggle/input/eyds-base-dataset
Debug: Submission directory: /kaggle/working/
Debug: Listing files in /kaggle/input/eyds-base-dataset:
['census_block_loc.csv', 'Hyperlocal_Temperature_Monitoring_20250311.csv', 'Airquality_Unique_geocode_with_LatLong.xlsx', 'nyclion_25a', 'StreetAssessmentRating', 'USA_wind-speed_10m.tif', 'USA_power-density_10m.tif', 'Validation_data.csv', 'LSAT_8_221022', 'Training_data.csv', 'NYC_Cooling_Tower_Registrations_20250224.csv', 'AQ', 'Automated_Traffic_Volume_Counts_20250319.csv', 'USA_air-density_10m.tif', 'nclimgrid-monthly-202107.tif', 'Energy_and_Water_Data_Disclosure_for_Local_Law_84_2022__Data_for_Calendar_Year_2021__20250224.csv',

In [30]:
# Download OSM features for additional metrics

# 1. Land Use: using 'landuse' key
landuse_tags = {'landuse': ['residential', 'commercial', 'industrial', 'retail', 'forest', 'grass', 'recreation']}
print("Downloading land use features from OSM...")
landuse_features = ox.features.features_from_polygon(bbox_polygon, tags=landuse_tags)
# Ensure CRS is set; if not, set to EPSG:4326, then reproject
if landuse_features.crs is None:
    landuse_features.set_crs("EPSG:4326", inplace=True)
landuse_features = landuse_features.to_crs(target_crs)
print(f"Debug: Land use features shape: {landuse_features.shape}")
print(f"Debug: Land use features columns: {landuse_features.columns.tolist()}")

# 2. Road network (for intersections and road classification)
print("Downloading road network from OSM...")
G_roads = ox.graph_from_polygon(bbox_polygon, network_type='drive')
nodes, roads = ox.graph_to_gdfs(G_roads)
roads = roads.to_crs(target_crs)
print(f"Debug: Road network nodes shape: {nodes.shape}")
print(f"Debug: Road network edges shape: {roads.shape}")
print(f"Debug: Road network columns: {roads.columns.tolist()}")

# 3. Public Transit: bus stops, subway stations, tram stops
transit_tags = {'highway': 'bus_stop', 'railway': ['subway_station', 'tram_stop']}
print("Downloading transit features from OSM...")
transit_features = ox.features.features_from_polygon(bbox_polygon, tags=transit_tags)
if transit_features.crs is None:
    transit_features.set_crs("EPSG:4326", inplace=True)
transit_features = transit_features.to_crs(target_crs)
print(f"Debug: Transit features shape: {transit_features.shape}")
print(f"Debug: Transit features columns: {transit_features.columns.tolist()}")

# 4. Parking
parking_tags = {'amenity': 'parking'}
print("Downloading parking features from OSM...")
parking_features = ox.features.features_from_polygon(bbox_polygon, tags=parking_tags)
if parking_features.crs is None:
    parking_features.set_crs("EPSG:4326", inplace=True)
parking_features = parking_features.to_crs(target_crs)
print(f"Debug: Parking features shape: {parking_features.shape}")
print(f"Debug: Parking features columns: {parking_features.columns.tolist()}")

# 5. Pedestrian and cycle infrastructure
ped_cycle_tags = {'highway': ['footway', 'cycleway']}
print("Downloading pedestrian/cycle infrastructure features from OSM...")
ped_cycle_features = ox.features.features_from_polygon(bbox_polygon, tags=ped_cycle_tags)
if ped_cycle_features.crs is None:
    ped_cycle_features.set_crs("EPSG:4326", inplace=True)
ped_cycle_features = ped_cycle_features.to_crs(target_crs)
print(f"Debug: Pedestrian/cycle features shape: {ped_cycle_features.shape}")
print(f"Debug: Pedestrian/cycle features columns: {ped_cycle_features.columns.tolist()}")

# 6. Parks (for distance and density features)
park_tags = {'leisure': 'park'}
print("Downloading park features from OSM...")
parks_gdf = ox.features.features_from_polygon(bbox_polygon, tags=park_tags)
if parks_gdf.crs is None:
    parks_gdf.set_crs("EPSG:4326", inplace=True)
parks_gdf = parks_gdf.to_crs(target_crs)
print(f"Debug: Park features shape: {parks_gdf.shape}")
print(f"Debug: Park features columns: {parks_gdf.columns.tolist()}")

# 7. Water features
water_tags = {
    "waterway": ["river", "stream", "canal"],
    "natural": ["water", "coastline"],
    "water": ["lake", "river", "reservoir"],
    "landuse": ["reservoir"]
}
print("Downloading water features from OSM...")
water_features = ox.features.features_from_polygon(bbox_polygon, tags=water_tags)
if water_features.crs is None:
    water_features.set_crs("EPSG:4326", inplace=True)
water_features = water_features.to_crs(target_crs)
print(f"Debug: Water features shape: {water_features.shape}")
print(f"Debug: Water features columns: {water_features.columns.tolist()}")

# 8. Buildings (for SVF proxy)
building_tags = {'building': True}
print("Downloading building footprints from OSM for SVF...")
buildings_osm = ox.features.features_from_polygon(bbox_polygon, tags=building_tags)
if buildings_osm.crs is None:
    buildings_osm.set_crs("EPSG:4326", inplace=True)
buildings_osm = buildings_osm.to_crs(target_crs)
print(f"Debug: OSM Buildings shape: {buildings_osm.shape}")
print(f"Debug: OSM Buildings columns: {buildings_osm.columns.tolist()}")

Debug: Land use features shape: (1582, 80)
Debug: Land use features columns: ['geometry', 'hazard', 'landuse', 'man_made', 'name', 'ref', 'source', 'beauty', 'shop', 'addr:city', 'addr:postcode', 'addr:state', 'office', 'addr:housenumber', 'addr:street', 'residential', 'website', 'branch', 'opening_hours', 'phone', 'wheelchair', 'barrier', 'layer', 'operator', 'wikidata', 'image', 'industrial', 'wikipedia', 'operator:type', 'operator:wikidata', 'start_date', 'is_in', 'name:etymology:wikidata', 'area', 'traffic_calming', 'operator:short', 'ele', 'gnis:feature_id', 'plant:method', 'plant:output:electricity', 'plant:source', 'power', 'ref:US:EIA', 'plant:type', 'website:alternate', 'architect', 'architect:wikidata', 'architect:wikipedia', 'building', 'height', 'nycdoitt:bin', 'plant:output:steam', 'railway', 'area:highway', 'owner', 'alt_name', 'name:be', 'name:en', 'name:pl', 'name:ru', 'subject', 'subject:wikidata', 'subject:wikipedia', 'landcover', 'golf', 'leisure', 'sport', 'amenity'

In [31]:
# Reproject our train and validation GeoDataFrames to target CRS
print("Reprojecting training and validation GeoDataFrames to target CRS...")
train_gdf = train_gdf.to_crs(target_crs)
val_gdf = val_gdf.to_crs(target_crs)
print(f"Debug: Training GeoDataFrame CRS after reprojection: {train_gdf.crs}")
print(f"Debug: Validation GeoDataFrame CRS after reprojection: {val_gdf.crs}")

# Function to calculate minimum distance to features
def calculate_distance_features(points_gdf, features_gdf, feature_type):
    """Calculate the minimum distance from each point to any feature (used for roads, parks, water)."""
    if features_gdf.empty:
        print(f"Debug: {feature_type} GeoDataFrame is empty. Returning NaN distances.")
        return np.full(len(points_gdf), np.nan)
    distances = points_gdf.geometry.apply(lambda pt: features_gdf.geometry.distance(pt).min())
    print(f"Debug: Calculated {feature_type} distances. Sample: {distances[:5].tolist()}")
    return distances

# Function to calculate density features (roads, parks)
def calculate_density_features(points_gdf, features_gdf, feature_type, radius_list=[100, 250, 500, 1000]):
    """
    Calculate density features for a given feature type (roads or parks)
    using area/length ratios, Gaussian KDE, and weighted coverage scores.
    """
    results = {}
    
    # 1. Area/length ratio within different radii
    for radius in radius_list:
        buffers = points_gdf.geometry.buffer(radius)
        density_values = []
        for buffer in buffers:
            features_in_buffer = features_gdf.clip(buffer)
            if features_in_buffer.empty:
                density_values.append(0)
            else:
                if feature_type == 'roads':
                    # For roads, use total length over the buffer perimeter
                    feature_measure = features_in_buffer.geometry.length.sum()
                    buffer_measure = buffer.length
                else:
                    # For parks, use area ratio
                    feature_measure = features_in_buffer.geometry.area.sum()
                    buffer_measure = buffer.area
                density_values.append(feature_measure / buffer_measure)
        results[f'{feature_type}_ratio_{radius}m'] = density_values
        print(f"Debug: Calculated {feature_type}_ratio_{radius}m. Sample: {density_values[:5]}")

    # 2. Gaussian KDE of feature boundaries
    feature_points = []
    for geom in features_gdf.geometry:
        if geom.geom_type == 'LineString':
            feature_points.extend(list(geom.coords))
        elif geom.geom_type == 'MultiLineString':
            for line in geom.geoms:
                feature_points.extend(list(line.coords))
        elif geom.geom_type in ['Polygon', 'MultiPolygon']:
            if geom.geom_type == 'Polygon':
                feature_points.extend(list(geom.exterior.coords))
            else:
                for polygon in geom.geoms:
                    feature_points.extend(list(polygon.exterior.coords))
    if feature_points:
        feature_points = np.array(feature_points)
        kde = gaussian_kde(feature_points.T)
        point_coords = np.array([(p.x, p.y) for p in points_gdf.geometry])
        kde_values = kde(point_coords.T)
        results[f'{feature_type}_kde'] = kde_values
        print(f"Debug: Calculated {feature_type}_kde. Sample: {kde_values[:5].tolist()}")
    else:
        results[f'{feature_type}_kde'] = np.zeros(len(points_gdf))
        print(f"Debug: No {feature_type} points for KDE. Assigned zeros.")
    
    # 3. Weighted coverage score using centroids
    if feature_type == 'roads':
        centroids = np.array([(geom.centroid.x, geom.centroid.y) for geom in features_gdf.geometry])
        weights = features_gdf.geometry.length.values
    else:
        centroids = np.array([(geom.centroid.x, geom.centroid.y) for geom in features_gdf.geometry])
        weights = features_gdf.geometry.area.values
    if len(centroids) > 0:
        tree = cKDTree(centroids)
        point_coords = np.array([(p.x, p.y) for p in points_gdf.geometry])
        k = min(5, len(centroids))
        distances, indices = tree.query(point_coords, k=k)
        weighted_scores = []
        for dist, idx in zip(distances, indices):
            if np.isscalar(dist):
                score = (weights[idx] / (dist + 1)) if dist > 0 else weights[idx]
                weighted_scores.append(score)
            else:
                scores = weights[idx] / (dist + 1)
                weighted_scores.append(np.sum(scores))
        results[f'{feature_type}_weighted_score'] = weighted_scores
        print(f"Debug: Calculated {feature_type}_weighted_score. Sample: {weighted_scores[:5]}")
    else:
        results[f'{feature_type}_weighted_score'] = np.zeros(len(points_gdf))
        print(f"Debug: No {feature_type} centroids for weighted score. Assigned zeros.")
    
    return pd.DataFrame(results)

# Function to calculate water density features
def calculate_water_density_features(points_gdf, water_features, radius_list=[100, 200, 500, 1000]):
    """Calculate water density features using multiple approaches"""
    results = {}
    
    # 1. Water area ratio
    for radius in radius_list:
        buffers = points_gdf.geometry.buffer(radius)
        water_ratios = []
        for buff in buffers:
            water_in_buff = water_features.clip(buff)
            if water_in_buff.empty:
                water_ratios.append(0)
            else:
                ratio = water_in_buff.geometry.area.sum() / buff.area
                water_ratios.append(ratio)
        results[f'water_ratio_{radius}m'] = water_ratios
        print(f"Debug: Calculated water_ratio_{radius}m. Sample: {water_ratios[:5]}")

    # 2. Gaussian KDE of water bodies
    water_points = []
    for geom in water_features.geometry:
        if geom.geom_type == 'Polygon':
            water_points.extend(list(geom.exterior.coords))
        elif geom.geom_type == 'MultiPolygon':
            for polygon in geom.geoms:
                water_points.extend(list(polygon.exterior.coords))
    if water_points:
        water_points = np.array(water_points)
        kde = gaussian_kde(water_points.T)
        point_coords = np.array([(p.x, p.y) for p in points_gdf.geometry])
        kde_values = kde(point_coords.T)
        results['water_kde'] = kde_values
        print(f"Debug: Calculated water_kde. Sample: {kde_values[:5].tolist()}")
    else:
        results['water_kde'] = np.zeros(len(points_gdf))
        print("Debug: No water points for KDE. Assigned zeros.")

    # 3. Weighted water score
    water_centroids = np.array([(p.centroid.x, p.centroid.y) for p in water_features.geometry])
    if len(water_centroids) > 0:
        tree = cKDTree(water_centroids)
        point_coords = np.array([(p.x, p.y) for p in points_gdf.geometry])
        distances, indices = tree.query(point_coords, k=min(5, len(water_centroids)))
        water_areas = water_features.geometry.area.values
        weighted_scores = []
        for dist, idx in zip(distances, indices):
            if np.isscalar(dist):
                score = water_areas[idx] / (dist + 1) if dist > 0 else water_areas[idx]
                weighted_scores.append(score)
            else:
                score = np.sum(water_areas[idx] / (dist + 1))
                weighted_scores.append(score)
        results['weighted_water_score'] = weighted_scores
        print(f"Debug: Calculated weighted_water_score. Sample: {weighted_scores[:5]}")
    else:
        results['weighted_water_score'] = np.zeros(len(points_gdf))
        print("Debug: No water centroids for weighted score. Assigned zeros.")
    
    return pd.DataFrame(results)

# Function to calculate land use composition
def calculate_landuse_composition(points_gdf, landuse_gdf, radius_list=[100, 200, 500, 1000]):
    """Calculate the proportional area of selected land use categories within buffers"""
    results = {}
    categories = ['residential', 'commercial', 'industrial', 'retail', 'forest', 'grass']
    for radius in radius_list:
        buffers = points_gdf.geometry.buffer(radius)
        for cat in categories:
            ratios = []
            for buff in buffers:
                subset = landuse_gdf[landuse_gdf.get('landuse') == cat]
                clipped = subset.clip(buff)
                if clipped.empty:
                    ratios.append(0)
                else:
                    ratios.append(clipped.geometry.area.sum() / buff.area)
            results[f'landuse_{cat}_ratio_{radius}m'] = ratios
            print(f"Debug: Calculated landuse_{cat}_ratio_{radius}m. Sample: {ratios[:5]}")
    return pd.DataFrame(results)

# Function to calculate transit accessibility
def calculate_transit_accessibility(points_gdf, transit_gdf, radius_list=[100, 200, 500, 1000]):
    """Count transit features (bus stops, subway stations, tram stops) within buffers"""
    results = {}
    for radius in radius_list:
        counts = []
        for pt in points_gdf.geometry:
            buff = pt.buffer(radius)
            count = transit_gdf[transit_gdf.within(buff)].shape[0]
            counts.append(count)
        results[f'transit_count_{radius}m'] = counts
        print(f"Debug: Calculated transit_count_{radius}m. Sample: {counts[:5]}")
    return pd.DataFrame(results)

# Function to calculate road classification
def calculate_road_classification(points_gdf, roads_gdf, radius_list=[100, 200, 500, 1000]):
    """Calculate the ratio of major to minor road lengths within buffers"""
    major_types = ['motorway', 'trunk', 'primary']
    minor_types = ['secondary', 'tertiary', 'residential', 'service']
    results = {}
    for radius in radius_list:
        major_length = []
        minor_length = []
        total_length = []
        for pt in points_gdf.geometry:
            buff = pt.buffer(radius)
            clipped = roads_gdf[roads_gdf.intersects(buff)].copy()
            if clipped.empty:
                major_length.append(0)
                minor_length.append(0)
                total_length.append(0)
            else:
                clipped['intersected'] = clipped.geometry.intersection(buff)
                lengths = clipped['intersected'].length
                tot = lengths.sum()
                total_length.append(tot)
                major_length.append(lengths[clipped['highway'].isin(major_types)].sum())
                minor_length.append(lengths[clipped['highway'].isin(minor_types)].sum())
        major_ratio = [maj/total if total > 0 else 0 for maj, total in zip(major_length, total_length)]
        minor_ratio = [minr/total if total > 0 else 0 for minr, total in zip(minor_length, total_length)]
        results[f'road_major_ratio_{radius}m'] = major_ratio
        results[f'road_minor_ratio_{radius}m'] = minor_ratio
        print(f"Debug: Calculated road_major_ratio_{radius}m. Sample: {major_ratio[:5]}")
        print(f"Debug: Calculated road_minor_ratio_{radius}m. Sample: {minor_ratio[:5]}")
    return pd.DataFrame(results)

# Function to calculate parking density
def calculate_parking_density(points_gdf, parking_gdf, radius_list=[100, 200, 500, 1000]):
    """Calculate the ratio of parking area to buffer area within buffers"""
    results = {}
    for radius in radius_list:
        area_ratios = []
        for pt in points_gdf.geometry:
            buff = pt.buffer(radius)
            clipped = parking_gdf.clip(buff)
            if clipped.empty:
                area_ratios.append(0)
            else:
                area_ratios.append(clipped.geometry.area.sum() / buff.area)
        results[f'parking_area_ratio_{radius}m'] = area_ratios
        print(f"Debug: Calculated parking_area_ratio_{radius}m. Sample: {area_ratios[:5]}")
    return pd.DataFrame(results)

# Function to calculate pedestrian/cycle infrastructure counts
def calculate_ped_cycle_infra(points_gdf, infra_gdf, radius_list=[100, 200, 500, 1000]):
    """Count pedestrian/cycle features (footways, cycleways) within buffers"""
    results = {}
    for radius in radius_list:
        counts = []
        for pt in points_gdf.geometry:
            buff = pt.buffer(radius)
            count = infra_gdf[infra_gdf.within(buff)].shape[0]
            counts.append(count)
        results[f'ped_cycle_count_{radius}m'] = counts
        print(f"Debug: Calculated ped_cycle_count_{radius}m. Sample: {counts[:5]}")
    return pd.DataFrame(results)

# Function to clean the 'height' column
def clean_height_column(height_series):
    """
    Clean the 'height' column by extracting the first valid numeric value from each entry.
    If the value cannot be converted to a float, return NaN.
    
    Parameters:
        height_series (pd.Series): Series containing height values as strings.
    
    Returns:
        pd.Series: Series with cleaned numeric height values (or NaN for invalid entries).
    """
    def extract_first_number(value):
        if pd.isna(value) or not isinstance(value, str):
            return np.nan
        # Split the string on non-numeric characters and take the first valid number
        parts = value.split()
        for part in parts:
            try:
                # Try to convert the part to a float
                return float(part)
            except ValueError:
                continue
        return np.nan

    return height_series.apply(extract_first_number)

# Apply the cleaning function to the 'height' column of buildings_osm
print("Cleaning 'height' column in buildings_osm...")
buildings_osm['height'] = clean_height_column(buildings_osm['height'])
print(f"Debug: Sample of cleaned 'height' values: {buildings_osm['height'].head().tolist()}")


# Function to calculate Sky View Factor (SVF) proxy
def calculate_svf(points_gdf, buildings_gdf, radius=100):
    """
    A simple proxy for Sky View Factor (SVF) using average building height in the buffer.
    NOTE: True SVF computation requires a 3D analysis.
    
    Parameters:
        points_gdf (gpd.GeoDataFrame): GeoDataFrame containing points.
        buildings_gdf (gpd.GeoDataFrame): GeoDataFrame containing building footprints.
        radius (float): Buffer radius in meters (default: 100).
    
    Returns:
        pd.DataFrame: DataFrame with 'svf_100m' column containing SVF proxy values.
    """
    svf_values = []
    for pt in points_gdf.geometry:
        buff = pt.buffer(radius)
        clipped = buildings_gdf[buildings_gdf.intersects(buff)]
        if clipped.empty:
            svf_values.append(1)  # Full sky visible
        else:
            # Ensure the 'height' column is numeric, coercing invalid values to NaN
            heights = pd.to_numeric(clipped['height'], errors='coerce')
            # Compute the mean height, ignoring NaN values
            avg_height = heights.mean() if not heights.isna().all() else 0
            # Crude proxy: assume maximum impact if avg height is 100 m
            svf = 1 - (avg_height / 100) if avg_height > 0 else 1
            svf = max(min(svf, 1), 0)  # Ensure SVF is between 0 and 1
            svf_values.append(svf)
    print(f"Debug: Calculated svf_100m. Sample: {svf_values[:5]}")
    return pd.DataFrame({'svf_100m': svf_values})

Reprojecting training and validation GeoDataFrames to target CRS...
Debug: Training GeoDataFrame CRS after reprojection: EPSG:32618
Debug: Validation GeoDataFrame CRS after reprojection: EPSG:32618
Cleaning 'height' column in buildings_osm...
Debug: Sample of cleaned 'height' values: [nan, nan, nan, nan, nan]


In [32]:
# Distance features
print("Computing distance features...")
train_gdf['dist_to_road'] = calculate_distance_features(train_gdf, roads, 'roads')
train_gdf['dist_to_park'] = calculate_distance_features(train_gdf, parks_gdf, 'parks')
train_gdf['dist_to_water'] = calculate_distance_features(train_gdf, water_features, 'water')

# Road density features
print("Computing road density features...")
train_road_density = calculate_density_features(train_gdf, roads, 'roads', radius_list=[100, 250, 500, 1000])

# Park density features
print("Computing park density features...")
train_park_density = calculate_density_features(train_gdf, parks_gdf, 'parks', radius_list=[100, 250, 500, 1000])

# Water density features
print("Computing water density features...")
train_water_density = calculate_water_density_features(train_gdf, water_features, radius_list=[100, 200, 500, 1000])

# Land use composition
print("Computing land use composition...")
train_landuse = calculate_landuse_composition(train_gdf, landuse_features, radius_list=[100, 200, 500, 1000])

# Transit accessibility
print("Computing transit accessibility...")
train_transit = calculate_transit_accessibility(train_gdf, transit_features, radius_list=[100, 200, 500, 1000])

# Road classification
print("Computing road classification metrics...")
train_roads_classification = calculate_road_classification(train_gdf, roads, radius_list=[100, 200, 500, 1000])

# Parking density
print("Computing parking density...")
train_parking = calculate_parking_density(train_gdf, parking_features, radius_list=[100, 200, 500, 1000])

# Pedestrian/cycle infrastructure
print("Computing pedestrian/cycle infrastructure counts...")
train_pedcycle = calculate_ped_cycle_infra(train_gdf, ped_cycle_features, radius_list=[100, 200, 500, 1000])

# Sky View Factor (SVF) proxy
print("Computing SVF proxy...")
train_svf = calculate_svf(train_gdf, buildings_osm, radius=100)

# Combine all training features
train_features = pd.concat([
    train_df,  # Original training data
    train_gdf[['dist_to_road', 'dist_to_park', 'dist_to_water']],
    train_road_density,
    train_park_density,
    train_water_density,
    train_landuse,
    train_transit,
    train_roads_classification,
    train_parking,
    train_pedcycle,
    train_svf
], axis=1)

# Debug: Print final training features
print(f"Debug: Final training features shape: {train_features.shape}")
print(f"Debug: Final training features columns: {train_features.columns.tolist()}")

print("Computing features for validation data...")

Computing distance features...
Debug: Calculated roads distances. Sample: [2.9412112687233485, 2.7457293684087367, 3.079848808961615, 3.18064100377361, 2.529600119972246]
Debug: Calculated parks distances. Sample: [114.7574390451531, 111.43838887284875, 106.88142335213566, 102.34355593461248, 98.97048796965417]
Debug: Calculated water distances. Sample: [1098.8890342819561, 1094.0458176032444, 1089.1470197936005, 1083.8964007340792, 1078.7476527028734]
Computing road density features...
Debug: Calculated roads_ratio_100m. Sample: [1.365800090281002, 1.3468925416304882, 1.317270513265029, 1.2837116802426227, 1.2553854936361735]
Debug: Calculated roads_ratio_250m. Sample: [2.8442839353318803, 2.819071177613519, 2.778095498649643, 2.734918641178884, 2.722282671594379]
Debug: Calculated roads_ratio_500m. Sample: [5.74778868942367, 5.786810829621827, 5.820725274419021, 5.843088842478161, 5.855091705878081]
Debug: Calculated roads_ratio_1000m. Sample: [12.199447040401678, 12.187209885513449,

In [33]:
# Distance features
print("Computing distance features...")
val_gdf['dist_to_road'] = calculate_distance_features(val_gdf, roads, 'roads')
val_gdf['dist_to_park'] = calculate_distance_features(val_gdf, parks_gdf, 'parks')
val_gdf['dist_to_water'] = calculate_distance_features(val_gdf, water_features, 'water')

# Road density features
print("Computing road density features...")
val_road_density = calculate_density_features(val_gdf, roads, 'roads', radius_list=[100, 250, 500, 1000])

# Park density features
print("Computing park density features...")
val_park_density = calculate_density_features(val_gdf, parks_gdf, 'parks', radius_list=[100, 250, 500, 1000])

# Water density features
print("Computing water density features...")
val_water_density = calculate_water_density_features(val_gdf, water_features, radius_list=[100, 200, 500, 1000])

# Land use composition
print("Computing land use composition...")
val_landuse = calculate_landuse_composition(val_gdf, landuse_features, radius_list=[100, 200, 500, 1000])

# Transit accessibility
print("Computing transit accessibility...")
val_transit = calculate_transit_accessibility(val_gdf, transit_features, radius_list=[100, 200, 500, 1000])

# Road classification
print("Computing road classification metrics...")
val_roads_classification = calculate_road_classification(val_gdf, roads, radius_list=[100, 200, 500, 1000])

# Parking density
print("Computing parking density...")
val_parking = calculate_parking_density(val_gdf, parking_features, radius_list=[100, 200, 500, 1000])

# Pedestrian/cycle infrastructure
print("Computing pedestrian/cycle infrastructure counts...")
val_pedcycle = calculate_ped_cycle_infra(val_gdf, ped_cycle_features, radius_list=[100, 200, 500, 1000])

# Sky View Factor (SVF) proxy
print("Computing SVF proxy...")
val_svf = calculate_svf(val_gdf, buildings_osm, radius=100)

# Combine all validation features
val_features = pd.concat([
    val_df,  # Original validation data
    val_gdf[['dist_to_road', 'dist_to_park', 'dist_to_water']],
    val_road_density,
    val_park_density,
    val_water_density,
    val_landuse,
    val_transit,
    val_roads_classification,
    val_parking,
    val_pedcycle,
    val_svf
], axis=1)

# Debug: Print final validation features
print(f"Debug: Final validation features shape: {val_features.shape}")
print(f"Debug: Final validation features columns: {val_features.columns.tolist()}")

# Output paths
TRAIN_OUTPUT_CSV = os.path.join(sub_dir, "Training_data_with_osmnx_features.csv")
VALIDATION_OUTPUT_CSV = os.path.join(sub_dir, "Validation_data_with_osmnx_features.csv")

# Save the results
print("Saving datasets with OSMnx features to CSV...")
train_features.to_csv(TRAIN_OUTPUT_CSV, index=False)
val_features.to_csv(VALIDATION_OUTPUT_CSV, index=False)

# Debug: Print the final confirmation messages with file paths
print(f"Debug: Training data with OSMnx features saved to: {TRAIN_OUTPUT_CSV}")
print(f"Debug: Validation data with OSMnx features saved to: {VALIDATION_OUTPUT_CSV}")

print(f"All OSMnx features saved to:\n{TRAIN_OUTPUT_CSV}\n{VALIDATION_OUTPUT_CSV}")

Computing distance features...
Debug: Calculated roads distances. Sample: [3.647876415705709, 3.5239522892795794, 25.18161353827356, 8.768130412194738, 1.9466313184579462]
Debug: Calculated parks distances. Sample: [5.7148525251274815, 5.915046914111067, 51.076863464043505, 45.24147087777342, 124.25653283940028]
Debug: Calculated water distances. Sample: [486.61576188110575, 511.4693831629873, 166.30071268301947, 568.389337551192, 303.0409041460204]
Computing road density features...
Debug: Calculated roads_ratio_100m. Sample: [0.8794686628812861, 0.7011531878714392, 0.9951959750414997, 0.6901863652174332, 0.7028413453864903]
Debug: Calculated roads_ratio_250m. Sample: [2.1547982569692294, 2.1733665991306483, 1.641536437915509, 2.277749580979602, 2.2276244593784083]
Debug: Calculated roads_ratio_500m. Sample: [4.909805310737827, 5.032388827720429, 3.0522341650296596, 5.188331473805563, 3.702936130185307]
Debug: Calculated roads_ratio_1000m. Sample: [7.911160293619278, 7.964211396729064